# Initialization

In [1]:
import os

os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["OMP_NUM_THREADS"] = "2"

In [2]:
%pip install implicit


[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
import logging

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [4]:
%matplotlib inline
%config InlineBackend.figure_format = 'png'
%config InlineBackend.figure_format = 'retina'

# Загрузка данных

In [5]:
items = pd.read_parquet("/workspaces/mle-recsys-start/items.par")
events = pd.read_parquet("/workspaces/mle-recsys-start/events.par")

In [6]:
display(items.head())

,book_id,author,title,description,genre_and_votes,num_pages,average_rating,ratings_count,text_reviews_count,publisher,...,genre_and_votes_dict.Young Adult-Young Adult Fantasy,genre_and_votes_dict.Young Adult-Young Adult Historical Fiction,genre_and_votes_dict.Young Adult-Young Adult Paranormal,genre_and_votes_dict.Young Adult-Young Adult Romance,genre_and_votes_dict.Young Adult-Young Adult Science Fiction,genre_and_votes_dict.k,genre_and_votes_dict.nd Grade,genre_and_votes_dict.st Century,genre_and_votes_dict.st Grade,genre_and_votes_dict.th Century
3.0,6066819,Jennifer Weiner,Best Friends Forever,Addie Downs and Valerie Adler were eight when ...,"{'Womens Fiction-Chick Lit': 739, 'Fiction': 442}",368,3.49,51184,3282,Atria Books,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6.0,378460,Michael Halberstam,The Wanting of Levine,,"{'Politics': 1, 'Humor': 1}",<NA>,4.38,12,4,Berkley Publishing Group,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
15.0,89375,"Don Piper, Cecil Murphey",90 Minutes in Heaven: A True Story of Death an...,As he is driving home from a minister's confer...,"{'Christian': 395, 'Nonfiction': 392, 'Religio...",<NA>,3.91,68157,2885,,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
16.0,89376,Randy Alcorn,Heaven,What is Heaven really going to be like? What w...,"{'Christian': 225, 'Religion-Theology': 154, '...",533,4.26,7345,566,,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
17.0,89377,Jennifer L. Holm,Penny from Heaven,It's 1953 and 11-year-old Penny dreams of a su...,"{'Historical-Historical Fiction': 284, 'Childr...",288,3.98,6949,615,Random House Books for Young Readers,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [7]:
display(events.head())

,user_id,book_id,started_at,read_at,is_read,rating,is_reviewed,started_at_month
0,8842281e1d1347389f2ab93d60773d4d,22034,2015-07-12,2015-07-17,True,5,False,2015-07-01
1,8842281e1d1347389f2ab93d60773d4d,22318578,2015-06-07,2015-08-09,True,5,True,2015-06-01
2,8842281e1d1347389f2ab93d60773d4d,22551730,2015-06-24,2015-07-11,True,4,True,2015-06-01
3,8842281e1d1347389f2ab93d60773d4d,22816087,2015-09-27,2015-11-04,True,5,True,2015-09-01
4,8842281e1d1347389f2ab93d60773d4d,20527133,2015-05-05,2015-09-06,True,0,False,2015-05-01


# Разбиение с учётом хронологии

Рекомендательные системы на практике работают с учётом хронологии. Поэтому поток событий для тренировки и валидации полезно делить на то, что уже случилось, и что ещё случится. Это позволяет проводить валидацию на тех же пользователях, на которых тренировались, но на их событиях в будущем.

# === Знакомство: "холодный" старт

In [8]:
# зададим точку разбиения
train_test_global_time_split_date = pd.to_datetime("2017-08-01")

train_test_global_time_split_idx = (
    events["started_at"] < train_test_global_time_split_date
)

events_train = events[train_test_global_time_split_idx]
events_test = events[~train_test_global_time_split_idx]

# количество пользователей в train и test
users_train = events_train["user_id"].drop_duplicates()
users_test = events_test["user_id"].drop_duplicates()

# количество пользователей, которые есть и в train, и в test
common_users = users_train[users_train.isin(users_test)]

print(len(users_train), len(users_test), len(common_users))

536293 138148 131924


In [9]:
cold_users = users_test[~users_test.isin(users_train)]

print(len(cold_users))

6224


In [10]:
from sklearn.preprocessing import MinMaxScaler

top_pop_start_date = pd.to_datetime("2015-01-01").date()

item_popularity = events_train \
    .query("started_at >= @top_pop_start_date") \
    .groupby(["book_id"]).agg(users=("user_id", "nunique"), avg_rating=("rating", "mean")).reset_index()

# нормализация пользователей и среднего рейтинга, требуется для их приведения к одному масштабу
scaler = MinMaxScaler()
item_popularity[["users_norm", "avg_rating_norm"]] = scaler.fit_transform(
    item_popularity[["users", "avg_rating"]]
)

# вычисляем popularity_score, как скор популярности со штрафом за низкий рейтинг
item_popularity["popularity_score"] = (
    item_popularity["users_norm"] * item_popularity["avg_rating_norm"]
)

# Сортируем по убыванию popularity_score
item_popularity = item_popularity.sort_values(
    by="popularity_score",
    ascending=False
)

# Выбираем топ-100 книг с рейтингом не ниже 4
top_k_pop_items = item_popularity[
    item_popularity["avg_rating"] >= 4
].head(100)

In [11]:
# добавляем информацию о книгах
top_k_pop_items = top_k_pop_items.merge(
    items.set_index("book_id")[["author", "title", "genre_and_votes", "publication_year"]], on="book_id")

with pd.option_context('display.max_rows', 100):
    display(top_k_pop_items[["book_id", "author", "title", "publication_year", "users", "avg_rating", "popularity_score", "genre_and_votes"]])

,book_id,author,title,publication_year,users,avg_rating,popularity_score,genre_and_votes
0,18007564,Andy Weir,The Martian,2014,21647,4.069663,0.397052,"{'Science Fiction': 11966, 'Fiction': 8430}"
1,16096824,Sarah J. Maas,A Court of Thorns and Roses (A Court of Thorns...,2015,17819,4.065043,0.326464,"{'Fantasy': 14326, 'Young Adult': 4662, 'Roman..."
2,3,"J.K. Rowling, Mary GrandPré",Harry Potter and the Sorcerer's Stone (Harry P...,1997,16634,4.331249,0.324709,"{'Fantasy': 59818, 'Fiction': 17918, 'Young Ad..."
3,15881,"J.K. Rowling, Mary GrandPré",Harry Potter and the Chamber of Secrets (Harry...,1999,14571,4.163681,0.273431,"{'Fantasy': 50130, 'Young Adult': 15202, 'Fict..."
4,17927395,Sarah J. Maas,A Court of Mist and Fury (A Court of Thorns an...,2016,13026,4.443191,0.260846,"{'Fantasy': 10186, 'Romance': 3346, 'Young Adu..."
5,5,"J.K. Rowling, Mary GrandPré",Harry Potter and the Prisoner of Azkaban (Harr...,2004,13231,4.303907,0.256646,"{'Fantasy': 49784, 'Young Adult': 15393, 'Fict..."
6,21853621,Kristin Hannah,The Nightingale,2015,13011,4.313197,0.252923,"{'Historical-Historical Fiction': 10105, 'Fict..."
7,13206900,Marissa Meyer,"Winter (The Lunar Chronicles, #4)",2015,13230,4.220937,0.251679,"{'Fantasy': 4835, 'Young Adult': 4672, 'Scienc..."
8,9969571,Ernest Cline,Ready Player One,2011,12679,4.041801,0.230960,"{'Science Fiction': 10560, 'Fiction': 7713, 'Y..."
9,6,"J.K. Rowling, Mary GrandPré",Harry Potter and the Goblet of Fire (Harry Pot...,2002,11953,4.274408,0.230265,"{'Fantasy': 48257, 'Young Adult': 15483, 'Fict..."


In [12]:
cold_users_events_with_recs = \
    events_test[events_test["user_id"].isin(cold_users)] \
    .merge(
        top_k_pop_items[["book_id", "avg_rating"]],
        on="book_id",
        how="left"
    )

cold_user_items_no_avg_rating_idx = (
    cold_users_events_with_recs["avg_rating"].isnull()
)

cold_user_recs = cold_users_events_with_recs[
    ~cold_user_items_no_avg_rating_idx
][["user_id", "book_id", "rating", "avg_rating"]]

In [13]:
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error

# Рассчитываем RMSE
rmse = np.sqrt(
    mean_squared_error(
        cold_user_recs["rating"],
        cold_user_recs["avg_rating"]
    )
)

# Рассчитываем MAE
mae = mean_absolute_error(
    cold_user_recs["rating"],
    cold_user_recs["avg_rating"]
)

print(f"RMSE: {rmse:.2f}")
print(f"MAE: {mae:.2f}")


RMSE: 1.65
MAE: 1.12


In [14]:
# посчитаем покрытие холодных пользователей рекомендациями
cold_users_hit_ratio = cold_users_events_with_recs.groupby("user_id").agg(hits=("avg_rating", lambda x: (~x.isnull()).mean()))

print(f"Доля пользователей без релевантных рекомендаций: {(cold_users_hit_ratio == 0).mean().iat[0]:.2f}")
print(f"Среднее покрытие пользователей: {cold_users_hit_ratio[cold_users_hit_ratio != 0].mean().iat[0]:.2f}")

Доля пользователей без релевантных рекомендаций: 0.83
Среднее покрытие пользователей: 0.57


# === Знакомство: первые персональные рекомендации

In [15]:
from surprise import Dataset, Reader
from surprise import SVD

# используем Reader из библиотеки surprise для преобразования событий (events)
# в формат, необходимый surprise
reader = Reader(rating_scale=(1, 5))
surprise_train_set = Dataset.load_from_df(events_train[['user_id', 'book_id', 'rating']], reader)
surprise_train_set = surprise_train_set.build_full_trainset()

# инициализируем модель
svd_model = SVD(n_factors=100, random_state=0)

# обучаем модель
svd_model.fit(surprise_train_set)

In [16]:
surprise_test_set = list(events_test[['user_id', 'book_id', 'rating']].itertuples(index=False))

# получаем рекомендации для тестовой выборки
svd_predictions = svd_model.test(surprise_test_set)

In [17]:
from surprise import accuracy

rmse = accuracy.rmse(svd_predictions)
mae = accuracy.mae(svd_predictions)
                     
print(rmse, mae)

RMSE: 1.3469
MAE:  0.9899
1.3468989130561508 0.9898677202113106


In [18]:
from surprise import NormalPredictor, SVD, accuracy

# Фиксируем генератор случайных чисел
np.random.seed(0)

# 1. Случайная модель NormalPredictor
random_model = NormalPredictor()
random_model.fit(surprise_train_set)

random_predictions = random_model.test(surprise_test_set)

# Рассчитываем MAE случайной модели
random_mae = accuracy.mae(random_predictions, verbose=False)

# 2. Модель SVD
svd_model = SVD(random_state=0)
svd_model.fit(surprise_train_set)

svd_predictions = svd_model.test(surprise_test_set)

# Рассчитываем MAE модели SVD
svd_mae = accuracy.mae(svd_predictions, verbose=False)

# 3. Процент превышения MAE
mae_difference_percent = (random_mae - svd_mae) / svd_mae * 100

print(f"MAE NormalPredictor: {random_mae:.4f}")
print(f"MAE SVD: {svd_mae:.4f}")
print(f"MAE случайной модели выше на: {round(mae_difference_percent)}%")

MAE NormalPredictor: 1.4945
MAE SVD: 0.9899
MAE случайной модели выше на: 51%


Факультативное задание

Задание. Удалите из events события для редких айтемов — таких, с которыми взаимодействовало менее N пользователей. Возьмите небольшое N, например 2–3 пользователя. Получите рекомендации, посчитайте метрики, оцените, как они изменились. Подумайте, с чем могут быть связаны такие изменения.

In [19]:
# Минимальное количество уникальных пользователей
N = 3

# Считаем количество пользователей для каждой книги
item_user_counts = (
    events_train.groupby("book_id")["user_id"]
    .nunique()
)

# Оставляем книги с количеством пользователей >= N
popular_items = item_user_counts[
    item_user_counts >= N
].index

# Фильтруем обучающие события
events_train_filtered = events_train[
    events_train["book_id"].isin(popular_items)
].copy()

print(f"Событий до фильтрации: {len(events_train):,}")
print(f"Событий после фильтрации: {len(events_train_filtered):,}")

print(
    f"Удалено событий: "
    f"{len(events_train) - len(events_train_filtered):,}"
)

print(
    f"Удалено книг: "
    f"{events_train['book_id'].nunique() - events_train_filtered['book_id'].nunique():,}"
)

Событий до фильтрации: 12,406,925
Событий после фильтрации: 12,402,318
Удалено событий: 4,607
Удалено книг: 3,176


In [20]:
# Средний рейтинг и количество пользователей
item_stats = (
    events_train_filtered
    .groupby("book_id")
    .agg(
        avg_rating=("rating", "mean"),
        user_count=("user_id", "nunique")
    )
    .reset_index()
)

# Топ-100 популярных книг
top_k_pop_items_filtered = (
    item_stats
    .sort_values(
        by=["user_count", "avg_rating"],
        ascending=[False, False]
    )
    .head(100)
)

top_k_pop_items_filtered.head()

,book_id,avg_rating,user_count
35094,22557272,3.522489,44443
27308,11870085,4.074039,43855
27202,11735983,3.657509,32506
28668,13335037,3.821637,30494
23373,7260188,3.691513,29998


In [21]:

from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error
)

# Объединяем тестовые события с рекомендациями
events_with_recs_filtered = events_test.merge(
    top_k_pop_items_filtered[["book_id", "avg_rating"]],
    on="book_id",
    how="left"
)

# Оставляем события, для которых есть прогноз
recs_filtered = events_with_recs_filtered.dropna(
    subset=["rating", "avg_rating"]
).copy()

# Рассчитываем метрики
rmse_filtered = np.sqrt(
    mean_squared_error(
        recs_filtered["rating"],
        recs_filtered["avg_rating"]
    )
)

mae_filtered = mean_absolute_error(
    recs_filtered["rating"],
    recs_filtered["avg_rating"]
)

print(f"RMSE после фильтрации: {rmse_filtered:.4f}")
print(f"MAE после фильтрации: {mae_filtered:.4f}")

RMSE после фильтрации: 1.5221
MAE после фильтрации: 1.0749


In [22]:
hit_ratio_filtered = (
    events_with_recs_filtered
    .groupby("user_id")["avg_rating"]
    .apply(lambda x: x.notna().mean())
    .mean()
)

print(f"Hit Ratio: {hit_ratio_filtered:.4f}")

Hit Ratio: 0.1044


In [23]:
# Исходные рекомендации
events_with_recs_before = events_test.merge(
    top_k_pop_items[["book_id", "avg_rating"]],
    on="book_id",
    how="left"
)

recs_before = events_with_recs_before.dropna(
    subset=["rating", "avg_rating"]
)

rmse_before = np.sqrt(
    mean_squared_error(
        recs_before["rating"],
        recs_before["avg_rating"]
    )
)

mae_before = mean_absolute_error(
    recs_before["rating"],
    recs_before["avg_rating"]
)

hit_ratio_before = (
    events_with_recs_before
    .groupby("user_id")["avg_rating"]
    .apply(lambda x: x.notna().mean())
    .mean()
)

comparison = pd.DataFrame({
    "Метрика": ["RMSE", "MAE", "Hit Ratio"],
    "До фильтрации": [
        rmse_before,
        mae_before,
        hit_ratio_before
    ],
    "После фильтрации": [
        rmse_filtered,
        mae_filtered,
        hit_ratio_filtered
    ]
})

comparison["Изменение"] = (
    comparison["После фильтрации"]
    - comparison["До фильтрации"]
)

display(comparison.round(4))

,Метрика,До фильтрации,После фильтрации,Изменение
0,RMSE,1.4644,1.5221,0.0577
1,MAE,0.9806,1.0749,0.0943
2,Hit Ratio,0.0952,0.1044,0.0092


Задание

Создайте функцию, которая позволит получить рекомендации для конкретного пользователя, используя описанный подход. Дополните прекод.

In [24]:
import pandas as pd

def get_recommendations_svd(user_id, all_items, events, model, include_seen=True, n=5):

    """Возвращает n рекомендаций для user_id"""

    # Получим список идентификаторов всех книг
    all_items = set(events['book_id'].unique())

    # Учитываем флаг, стоит ли уже прочитанные книги включать в рекомендации
    if include_seen:
        items_to_predict = list(all_items)
    else:
        # Получим список книг, которые пользователь уже прочитал
        seen_items = set(
            events[events['user_id'] == user_id]['book_id'].unique()
        )

        # Книги, которые пользователь ещё не читал
        items_to_predict = list(all_items - seen_items)

    # Получаем предсказания рейтингов для каждой книги
    predictions = [
        model.predict(user_id, book_id)
        for book_id in items_to_predict
    ]

    # Сортируем по убыванию предсказанного рейтинга
    recommendations = sorted(
        predictions,
        key=lambda x: x.est,
        reverse=True
    )[:n]

    return pd.DataFrame(
        [(pred.iid, pred.est) for pred in recommendations],
        columns=["book_id", "score"]
    )

In [25]:
recommendations = get_recommendations_svd(
    user_id=123,
    all_items=events['book_id'].unique(),
    events=events,
    model=svd_model,
    include_seen=False,
    n=5
)

display(recommendations)

,book_id,score
0,29844341,4.711874
1,54741,4.692017
2,22552026,4.668152
3,22037424,4.580056
4,17874649,4.577286


In [26]:
recommendations = get_recommendations_svd(
    user_id=1296647,
    all_items=events['book_id'].unique(),
    events=events,
    model=svd_model,
    include_seen=True,
    n=5
)

print(recommendations.iloc[2]['book_id'])

22552026.0


In [27]:
# выберем произвольного пользователя из тренировочной выборки ("прошлого")
user_id = events_train['user_id'].sample().iat[0]

print(f"user_id: {user_id}")

print("История (последние события, recent)")
user_history = (
    events_train
    .query("user_id == @user_id")
    .merge(items.set_index("book_id")[["author", "title", "genre_and_votes"]], on="book_id")
)
user_history_to_print = user_history[["author", "title", "started_at", "read_at", "rating", "genre_and_votes"]].tail(10)
display(user_history_to_print)

print("Рекомендации")
user_recommendations = get_recommendations_svd(user_id, items, events_train, svd_model)
user_recommendations = user_recommendations.merge(items[["book_id", "author", "title", "genre_and_votes"]], on="book_id")
display(user_recommendations)

user_id: 17f99167ae1c2a63c6ad52925d6bf2e3
История (последние события, recent)


,author,title,started_at,read_at,rating,genre_and_votes
14,K.A. Linde,"Avoiding Responsibility (Avoiding, #2)",2012-12-02,2012-12-02,4,"{'Romance': 223, 'New Adult': 145, 'Romance-Co..."
15,Colleen Hoover,"Hopeless (Hopeless, #1)",2013-02-05,2013-02-13,5,"{'Romance': 3351, 'New Adult': 2171, 'Young Ad..."
16,R.K. Lilley,"In Flight (Up in the Air, #1)",2012-11-20,2012-11-21,3,"{'Romance': 975, 'Erotica-BDSM': 704, 'Adult F..."
17,Katie Ashley,"The Proposition (The Proposition, #1)",2012-11-17,2012-11-17,5,"{'Romance': 804, 'Romance-Contemporary Romance..."
18,J. Sterling,"The Perfect Game (The Perfect Game, #1)",2012-11-14,2012-11-17,3,"{'New Adult': 613, 'Romance': 583, 'Sports-Spo..."
19,Lisa Renee Jones,"If I Were You (Inside Out, #1)",2012-11-12,2014-11-12,4,"{'Romance': 480, 'Adult Fiction-Erotica': 337,..."
20,Jessica Sorensen,The Coincidence of Callie & Kayden (The Coinci...,2012-12-13,2012-12-14,5,"{'Romance': 1557, 'New Adult': 1541, 'Contempo..."
21,Aleatha Romig,"Consequences (Consequences, #1)",2012-11-01,2015-01-18,5,"{'Dark': 581, 'Romance': 405, 'Sociology-Abuse..."
22,Nicole Williams,"Crash (Crash, #1)",2012-10-31,2012-11-02,4,"{'Romance': 933, 'Young Adult': 668, 'New Adul..."
23,Jennifer Probst,"The Marriage Trap (Marriage to a Billionaire, #2)",2012-10-15,2012-10-30,3,"{'Romance': 702, 'Romance-Contemporary Romance..."


Рекомендации


,book_id,score,author,title,genre_and_votes
0,2,5,"J.K. Rowling, Mary GrandPré",Harry Potter and the Order of the Phoenix (Har...,"{'Fantasy': 46485, 'Young Adult': 15194, 'Fict..."
1,33,5,J.R.R. Tolkien,"The Lord of the Rings (The Lord of the Rings, ...","{'Fantasy': 16354, 'Classics': 4836, 'Fiction'..."
2,976,5,Dan Brown,Deception Point,"{'Fiction': 4080, 'Thriller': 2842, 'Mystery':..."
3,6816890,5,José Luís Peixoto,Morreste-me,"{'Cultural-Portugal': 16, 'European Literature..."
4,7342071,5,"Juan Díaz Canales, Juanjo Guarnido, Anthya Flo...","Blacksad (Blacksad, #1-3)","{'Sequential Art-Graphic Novels': 627, 'Sequen..."


Факультативное задание

Добавьте в events события для нового пользователя, как если бы он прочитал те книги, которые нравятся вам. Получите рекомендации для этого пользователя. Оцените, насколько они релевантны вашим интересам.

In [28]:
import uuid
import gc

# ==========================================
# 1. Создаём нового пользователя
# ==========================================

new_user_id = uuid.uuid4().hex

print(f"ID нового пользователя: {new_user_id}")


# ==========================================
# 2. Выбираем 5 книг из каталога
# ==========================================

favorite_items = (
    events['book_id']
    .dropna()
    .drop_duplicates()
    .head(5)
    .tolist()
)

print(f"Выбранные книги: {favorite_items}")


# ==========================================
# 3. Создаём события нового пользователя
# ==========================================

new_events = pd.DataFrame({
    'user_id': [new_user_id] * len(favorite_items),
    'book_id': favorite_items,
    'rating': [5.0] * len(favorite_items)
})

print("\nСобытия нового пользователя:")
display(new_events)


# ==========================================
# 4. Подготавливаем данные для SVD
# ==========================================

# Используем только необходимые столбцы
svd_columns = ['user_id', 'book_id', 'rating']

# Убираем события без рейтинга
events_rated = events.loc[
    events['rating'].notna(),
    svd_columns
]

print(f"\nСобытий с рейтингами: {len(events_rated)}")


# ==========================================
# 5. Добавляем нового пользователя
# ==========================================

events_svd = pd.concat(
    [events_rated, new_events[svd_columns]],
    ignore_index=True
)

print(f"Количество событий до: {len(events_rated)}")
print(f"Количество событий после: {len(events_svd)}")


# ==========================================
# 6. Проверяем результат
# ==========================================

new_user_data = events_svd[
    events_svd['user_id'] == new_user_id
]

print("\nСобытия нового пользователя:")
display(new_user_data)

print(
    "\nКоличество книг нового пользователя:",
    len(new_user_data)
)


# ==========================================
# 7. Проверяем расход памяти
# ==========================================

memory_mb = (
    events_svd.memory_usage(deep=True).sum()
    / 1024**2
)

print(f"\nПамять events_svd: {memory_mb:.2f} MB")

gc.collect()

ID нового пользователя: 114e87ff88044d7bbf7b354df077721e
Выбранные книги: [22034, 22318578, 22551730, 22816087, 20527133]

События нового пользователя:


,user_id,book_id,rating
0,114e87ff88044d7bbf7b354df077721e,22034,5.0
1,114e87ff88044d7bbf7b354df077721e,22318578,5.0
2,114e87ff88044d7bbf7b354df077721e,22551730,5.0
3,114e87ff88044d7bbf7b354df077721e,22816087,5.0
4,114e87ff88044d7bbf7b354df077721e,20527133,5.0



Событий с рейтингами: 12887523


Количество событий до: 12887523
Количество событий после: 12887528

События нового пользователя:


,user_id,book_id,rating
12887523,114e87ff88044d7bbf7b354df077721e,22034,5.0
12887524,114e87ff88044d7bbf7b354df077721e,22318578,5.0
12887525,114e87ff88044d7bbf7b354df077721e,22551730,5.0
12887526,114e87ff88044d7bbf7b354df077721e,22816087,5.0
12887527,114e87ff88044d7bbf7b354df077721e,20527133,5.0



Количество книг нового пользователя: 5

Память events_svd: 688.27 MB


3

In [ ]:
# Книги, которые пользователь уже оценил
seen_items = set(new_events['book_id'])

# Все доступные книги
all_items = events_svd['book_id'].unique()

# Предсказываем рейтинги для непрочитанных книг
recommendations = []

for book_id in all_items:
    if book_id not in seen_items:
        prediction = svd_model.predict(
            uid=new_user_id,
            iid=book_id
        )

        recommendations.append({
            'book_id': book_id,
            'score': prediction.est
        })

# Выбираем 10 лучших рекомендаций
recommendations_df = (
    pd.DataFrame(recommendations)
    .sort_values('score', ascending=False)
    .head(10)
    .reset_index(drop=True)
)

display(recommendations_df)

,book_id,score
0,29844341,4.711874
1,54741,4.692017
2,22552026,4.668152
3,22037424,4.580056
4,17874649,4.577286
5,6898978,4.574685
6,24812,4.573624
7,25793186,4.571148
8,31422889,4.556868
9,580213,4.541967


: 

# === Базовые подходы: коллаборативная фильтрация

In [ ]:
import scipy
import sklearn.preprocessing

# перекодируем идентификаторы пользователей: 
# из имеющихся в последовательность 0, 1, 2, ...
user_encoder = sklearn.preprocessing.LabelEncoder()
user_encoder.fit(events["user_id"])
events_train["user_id_enc"] = user_encoder.transform(events_train["user_id"])
events_test["user_id_enc"] = user_encoder.transform(events_test["user_id"])

# перекодируем идентификаторы объектов: 
# из имеющихся в последовательность 0, 1, 2, ...
item_encoder = sklearn.preprocessing.LabelEncoder()
item_encoder.fit(items["book_id"])
items["item_id_enc"] = item_encoder.transform(items["book_id"])
events_train["item_id_enc"] = item_encoder.transform(events_train["book_id"])
events_test["item_id_enc"] = item_encoder.transform(events_test["book_id"])


In [ ]:
# Вариант 1: Прямо из полученной колонки
print(events_train["item_id_enc"].max())


43304


In [ ]:
matrix_size_gb = (
    len(user_encoder.classes_) *
    len(item_encoder.classes_)
) // (1024 ** 3)

print(int(matrix_size_gb))


21


In [ ]:
# создаём sparse-матрицу формата CSR 
user_item_matrix_train = scipy.sparse.csr_matrix((
    events_train["rating"],
    (events_train['user_id_enc'], events_train['item_id_enc'])),
    dtype=np.int8)


import sys

sum([sys.getsizeof(i) for i in user_item_matrix_train.data])/1024**3

0.28887123335152864

In [ ]:
from implicit.als import AlternatingLeastSquares

als_model = AlternatingLeastSquares(
    factors=50,
    iterations=10,
    regularization=0.05,
    random_state=0,
    num_threads=2,
    use_gpu=False
)

als_model.fit(user_item_matrix_train)

/workspaces/mle-recsys-start/env_recsys_start/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/workspaces/mle-recsys-start/env_recsys_start/lib/python3.14/site-packages/implicit/cpu/als.py:96: RuntimeWarning: OpenBLAS is configured to use 4 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()
100%|██████████| 50/50 [02:51<00:00,  3.42s/it]


: 

In [ ]:
def get_recommendations_als(user_item_matrix, model, user_id, user_encoder, item_encoder, include_seen=True, n=5):
    """
    Возвращает отранжированные рекомендации для заданного пользователя
    """
    user_id_enc = user_encoder.transform([user_id])[0]
    recommendations = model.recommend(
         user_id_enc, 
         user_item_matrix[user_id_enc], 
         filter_already_liked_items=not include_seen,
         N=n)
    recommendations = pd.DataFrame({"item_id_enc": recommendations[0], "score": recommendations[1]})
    recommendations["item_id"] = item_encoder.inverse_transform(recommendations["item_id_enc"])
    
    return recommendations

# === Базовые подходы: контентные рекомендации

# === Базовые подходы: валидация

# === Двухстадийный подход: метрики

# === Двухстадийный подход: модель

# === Двухстадийный подход: построение признаков